In [1]:
import warnings
warnings.simplefilter(action="ignore", category=RuntimeWarning)

import hddm
import kabuki
import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import arviz as az
import xarray as xr

print("The current version of kabuki is: ", kabuki.__version__)
print("The current version of HDDM is: ", hddm.__version__)
print("The current version of arviz is: ", az.__version__)

The current version of kabuki is:  0.6.5RC4
The current version of HDDM is:  1.0.1RC
The current version of arviz is:  0.15.1


In [2]:
data = hddm.load_csv(hddm.__path__[0] + "/examples/cavanagh_theta_nn.csv")
data = data[data['subj_idx'].isin([0,1,2,3,4])]
data.head()

,subj_idx,stim,rt,response,theta,dbs,conf
0,0,LL,1.21,1.0,0.656275,1,HC
1,0,WL,1.63,1.0,-0.327889,1,LC
2,0,WW,1.03,1.0,-0.480285,1,HC
3,0,WL,2.77,1.0,1.927427,1,LC
4,0,WW,1.14,0.0,-0.213236,1,HC


In [3]:
%time
model_reg = hddm.HDDMRegressor(data, "v ~ 1 + C(conf, Treatment('LC'))",include = ['v', 'a', 't', 'z'])
# note: setting save_name argument will delete the _temp*.db file
save_name = "test/hddmregressor_example"
model_reg_infdata = model_reg.sample(
    500, chains = 4, 
    return_infdata = True, save_name = save_name, 
    sample_prior = True, loglike = False, ppc = True)

CPU times: user 2 µs, sys: 1 µs, total: 3 µs
Wall time: 5.01 µs
No model attribute --> setting up standard HDDM
Set model to ddm


/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - fw)
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - fw)
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - fw)
/opt/conda/lib/python3.9/site-packages/scipy/optimize/_optimize.py:2309: RuntimeWarning: invalid value encountered in double_scalars
  tmp2 = (x - v) * (fx - fw)


hddm sampling elpased time:  42.295 s
Start converting to InferenceData...
Start generating posterior prediction...
[                  1%                  ] 7 of 500 complete in 0.5 sec
[-                 3%                  ] 15 of 500 complete in 1.1 sec
[-                 4%                  ] 23 of 500 complete in 1.6 sec
[--                6%                  ] 31 of 500 complete in 2.2 sec
[--                7%                  ] 39 of 500 complete in 2.7 sec
[---               9%                  ] 47 of 500 complete in 3.3 sec
[----             11%                  ] 55 of 500 complete in 3.8 sec
[----             12%                  ] 63 of 500 complete in 4.4 sec
[-----            14%                  ] 71 of 500 complete in 4.9 sec
[-----            15%                  ] 78 of 500 complete in 5.5 sec
[------           17%                  ] 86 of 500 complete in 6.0 sec
[-------          18%                  ] 93 of 500 complete in 6.5 sec
[-------          20%            

In [4]:
model_reg_infdata

Inference data with groups:
	> posterior
	> posterior_predictive
	> prior
	> observed_data

In [9]:
%time
save_name = "test/hddmregressor_example"
# loading the inference data is faster than loading the origin hddm class
# model_reg = hddm.load(save_name + ".hddm")
model_reg_infdata = az.from_netcdf(save_name + ".nc")

CPU times: user 2 µs, sys: 1e+03 ns, total: 3 µs
Wall time: 6.2 µs


In [10]:
def plot_rt_quantiles_corrected(infdata, quantiles=[0.1, 0.3, 0.5, 0.7, 0.9]):
    """
    Plot quantile-quantile comparison between observed and posterior predictive reaction times (RTs).
    Handles positive (correct/upper boundary) and negative (incorrect/lower boundary) RTs separately.
    
    Parameters
    ----------
    infdata : arviz.InferenceData
        InferenceData object containing observed_data and posterior_predictive groups.
    quantiles : list of float, optional
        Quantile levels to compute and plot (default: [0.1, 0.3, 0.5, 0.7, 0.9]).
    """
    
    # 1. Extract observed data
    obs_rt_all = infdata.observed_data["rt"]
    
    # 2. Extract posterior predictive data (stack chain and draw dimensions)
    pp_rt_all = infdata.posterior_predictive["rt"].stack(sample=("chain", "draw"))

    # 3. Automatically detect response type based on RT sign:
    #    RT > 0 -> Response 1 (upper boundary/correct)
    #    RT < 0 -> Response 0 (lower boundary/incorrect)
    #    Note: Using absolute values for quantile computation
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    
    # === Define two conditions: positive RTs and negative RTs ===
    conditions = [
        {"name": "Positive RTs (Upper Boundary)", "sign": 1},
        {"name": "Negative RTs (Lower Boundary)", "sign": -1}
    ]
    
    for i, cond in enumerate(conditions):
        ax = axes[i]
        sign = cond["sign"]
        
        # --- A. Process observed data ---
        # Filter: keep only data with the current sign
        if sign == 1:
            curr_obs = obs_rt_all.where(obs_rt_all > 0, drop=True)
        else:
            curr_obs = obs_rt_all.where(obs_rt_all < 0, drop=True)
            
        # Skip if no data (e.g., only positive responses)
        if curr_obs.size == 0:
            ax.text(0.5, 0.5, "No Data", ha='center', fontsize=12)
            ax.set_title(cond["name"])
            continue

        # Take absolute values and compute quantiles
        # [Correction] Using dim="obs_id" for observed data
        curr_obs_abs = np.abs(curr_obs)
        obs_qs = curr_obs_abs.quantile(quantiles, dim="obs_id")
        
        # --- B. Process posterior predictive data ---
        # xarray's where operation works similarly for posterior data
        if sign == 1:
            curr_pp = pp_rt_all.where(pp_rt_all > 0, drop=True)
        else:
            curr_pp = pp_rt_all.where(pp_rt_all < 0, drop=True)
            
        # Take absolute values
        curr_pp_abs = np.abs(curr_pp)
        
        # Compute quantiles
        # Note: Posterior data typically retains the 'sample' dimension
        #       and aggregates over obs_id (trial) dimension
        # [Correction] Identify the non-sample dimension name dynamically
        pp_dim_name = [d for d in curr_pp_abs.dims if d not in ["sample", "chain", "draw"]][0]
        pp_qs = curr_pp_abs.quantile(quantiles, dim=pp_dim_name)
        
        # Calculate HDI (uncertainty intervals)
        # pp_qs has shape: (n_quantiles, n_samples)
        # Compute statistics across the sample dimension (axis=1)
        pp_qs_np = pp_qs.values
        
        # Compute mean and 94% interval for each quantile
        pp_mean = np.nanmean(pp_qs_np, axis=1)
        pp_lower = np.nanpercentile(pp_qs_np, 3, axis=1)
        pp_upper = np.nanpercentile(pp_qs_np, 97, axis=1)
        
        # --- C. Plotting ---
        # Plot posterior predictive interval (fan)
        ax.fill_between(
            quantiles, pp_lower, pp_upper, 
            color='C1', alpha=0.5, label='94% Posterior PPI'
        )
        ax.plot(
            quantiles, pp_mean, 'o-', 
            color='C1', label='Posterior Mean'
        )
        
        # Plot observed data
        ax.plot(
            quantiles, obs_qs, 'x--', 
            color='k', markersize=10, label='Observed Data'
        )
        
        ax.set_title(cond["name"])
        ax.set_xlabel("Quantile")
        ax.set_xticks(quantiles)
        if i == 0: 
            ax.set_ylabel("Reaction Time (|s|)")
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [11]:
plot_rt_quantiles_corrected(model_reg_infdata)

<Figure size 1200x500 with 2 Axes>